# Agent 8 — Creator Intelligence

> Score a content creator across production quality, audio quality,
> delivery, hook strength, and brand safety. Per-video VLM scoring →
> aggregated scorecard with a strong-fit / moderate-fit / not-recommended
> recommendation.

## What's different about this agent

This one **doesn't use `/search` at all**. You hand it a list of public
video URLs (e.g. the creator's recent uploads), and it runs the VLM on
each one with a fixed multi-dimensional scoring prompt.

## Endpoints exercised

| Step | Endpoint |
|---|---|
| Score each video | `POST /vu/chat/completions` (×N videos) |


## Setup

You need:

1. A Memories.ai API key (`sk-mavi-...`) — get one at the [Memories.ai console](https://api-platform.memories.ai/stripe).
2. Python 3.10+ and the `requests` library (`pip install requests`).

Set the key as an environment variable before launching Jupyter, or paste it
inline in the cell below. The same key works across Visual Search, Visual
Intelligence, and Visual Agents — no separate auth per product.


In [ ]:
import os, json, time, requests

# ────────────────────────────────────────────────────────────────────────────
# Auth: the Memories.ai key is a single token used across every product.
# Pass it as the literal `Authorization` header value — no `Bearer` prefix.
# ────────────────────────────────────────────────────────────────────────────
API_KEY = os.environ.get("MEMORIES_API_KEY") or "sk-mavi-..."  # ← paste here if not using env
HEADERS = {"Authorization": API_KEY}

# Hosts: Visual Search and Visual Intelligence live on different domains.
VS_HOST  = "https://api.memories.ai/serve/api/v1"            # Visual Search
VLM_HOST = "https://mavi-backend.memories.ai/serve/api/v2"   # Visual Intelligence (VLM + Visual Agents)

# Default VLM model used for verification / reasoning. Other options include
# `qwen:qwen2.5-vl-72b-instruct`, `nova:amazon.nova-lite-v1:0`, etc. — see
# Memories.ai docs for the full list and per-token pricing.
VLM_MODEL = "gemini:gemini-2.5-flash"


In [ ]:
def vlm_complete(prompt, *, video_url=None, image_url=None, system=None,
                 model=VLM_MODEL, response_json=True, temperature=0.2,
                 max_tokens=1024):
    """Visual Intelligence — POST /vu/chat/completions.

    Calls a Video Language Model (Gemini by default) with text + an
    optional media reference. Returns the assistant's text reply, with
    markdown ```json fences stripped if `response_json=True`.

    Notes on the wire format:
      • `content` MUST be an array even for text-only prompts. A bare
        string is rejected with "Model input cannot be empty".
      • Gemini's response lives in choices[0]["text"]. Other providers
        (Qwen, Nova) use choices[0]["message"]["content"]. We accept both.
      • The endpoint can return HTTP 200 with status="errored" — surface
        that as a typed exception rather than letting JSON parsing fail.
    """
    content = [{"type": "text", "text": prompt}]
    if video_url:
        content.append({"type": "input_file", "file_uri": video_url, "mime_type": "video/mp4"})
    if image_url:
        content.append({"type": "input_file", "file_uri": image_url, "mime_type": "image/jpeg"})

    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": content})

    body = {
        "model": model,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
    }
    if response_json:
        # Tells Gemini to bias toward JSON output. Other providers ignore this.
        body["extra_body"] = {"metadata": {"response_mime_type": "application/json"}}

    r = requests.post(f"{VLM_HOST}/vu/chat/completions", headers=HEADERS,
                      json=body, timeout=180)
    r.raise_for_status()
    envelope = r.json()
    if envelope.get("status") == "errored" or envelope.get("error"):
        err = envelope.get("error") or {}
        raise RuntimeError(f"VLM error: {err.get('code')} {err.get('message')}")
    choices = envelope.get("choices") or []
    if not choices:
        raise RuntimeError(f"VLM returned no choices: {envelope}")
    # Two shapes observed in the wild.
    text = choices[0].get("text") or (choices[0].get("message") or {}).get("content", "")
    if response_json:
        text = _strip_json_fence(text)
    return text


def _strip_json_fence(text):
    """Gemini often wraps JSON output in ```json ... ``` fences even when
    response_mime_type=application/json. Trim them so json.loads() works."""
    if not text:
        return text
    s = text.strip()
    if s.startswith("```"):
        s = s.split("\n", 1)[1] if "\n" in s else s[3:]
        if s.endswith("```"):
            s = s[:-3].rstrip()
    return s


## Step 1 — declare the scoring schema

The schema is the contract between the VLM and your aggregation code.
Keep dimensions distinct and ask for the full 0-100 range (or the VLM
will cluster everything around 70-90).


In [ ]:
CREATOR_NAME = "@test_creator"

# Replace with the creator's recent videos. Public URLs only — the VLM
# needs to fetch them. For a real run you'd pull these from the creator's
# TikTok/YouTube/Instagram profile.
VIDEO_URLS = [
    "https://storage.googleapis.com/memories-test-data/test_1min.mp4",
    # add more...
]

SCORE_SCHEMA = (
    '{"production_quality": int, "audio_quality": int, "delivery": int, '
    '"hook_strength": int, "brand_safety": int, "content_style": str, '
    '"red_flags": [str], "notes": str}'
)


## Step 2 — score each video

We use a low temperature (0.1) so two calls on the same video give
similar scores, and ask the VLM to use the full range rather than
clustering everything at 70-90.


In [ ]:
SYSTEM = ("You are a creative director evaluating a content creator's video. "
          "Score each dimension on a 0-100 integer scale. Be strict — use the "
          "full range, not just 60-90. Reply strict JSON only.")

SCORE_PROMPT = (
    "Evaluate this video on six dimensions:\n"
    "- production_quality: lighting, framing, resolution, editing\n"
    "- audio_quality: clarity, background noise, music/voice balance\n"
    "- delivery: confidence, pacing, energy, authenticity\n"
    "- hook_strength: how compelling are the first 3 seconds\n"
    "- brand_safety: profanity, controversy, competitor mentions "
    "(100 = perfectly safe, 0 = unusable)\n"
    "- content_style: 1-3 word descriptor of the style (e.g. 'fast-cut tutorial')\n"
    f"Reply JSON only, matching: {SCORE_SCHEMA}"
)

per_video = []
for i, url in enumerate(VIDEO_URLS):
    print(f"[{i+1}/{len(VIDEO_URLS)}] scoring {url}")
    raw = vlm_complete(SCORE_PROMPT, video_url=url, system=SYSTEM,
                       temperature=0.1, max_tokens=512)
    try:
        score = json.loads(raw)
    except json.JSONDecodeError:
        score = {"raw": raw, "parse_error": True}
    per_video.append(score)
    print(f"  -> {score}\n")


## Step 3 — aggregate into a creator scorecard

Mean each dimension across the sample, then mean those into a single
overall score. Collect any red_flags into a creator-wide list.


In [ ]:
import statistics

def mean(values):
    clean = [v for v in values if isinstance(v, (int, float))]
    return round(statistics.mean(clean), 1) if clean else None

DIMS = ["production_quality", "audio_quality", "delivery",
        "hook_strength", "brand_safety"]
per_dim = {d: mean([s.get(d) for s in per_video if isinstance(s, dict)])
           for d in DIMS}

valid = [s for s in per_video if isinstance(s, dict) and "production_quality" in s]
overall = mean([statistics.mean([s.get(d, 0) for d in DIMS]) for s in valid])

red_flags = []
for s in per_video:
    if isinstance(s, dict):
        red_flags.extend(s.get("red_flags") or [])


## Step 4 — turn the scores into a recommendation

Brand-safety is the dominant signal — a single low brand-safety score
gates a creator regardless of how strong the rest looks.


In [ ]:
def recommendation(overall, brand_safety):
    if (brand_safety or 0) < 70:
        return "Not Recommended (brand-safety risk)"
    if (overall or 0) >= 75:
        return "Strong Fit"
    if (overall or 0) >= 60:
        return "Moderate Fit"
    return "Not Recommended"

scorecard = {
    "creator": CREATOR_NAME,
    "overall_score": overall,
    "per_dimension": per_dim,
    "video_count": len(per_video),
    "red_flags": red_flags,
    "recommendation": recommendation(overall, per_dim.get("brand_safety")),
}
print(json.dumps(scorecard, indent=2))


## Where to go next

- **Larger sample**: a real evaluation runs 10-20 videos per creator
  for stable scores. One outlier shouldn't sink a creator.
- **Time-trend tracking**: re-score the same creator monthly to spot
  quality drift.
- **Custom dimensions**: add `niche_fit` or `cta_strength` to the
  schema for campaign-specific scoring.
- **Cross-creator ranking**: run this notebook in a loop over a
  candidate pool, then sort by `overall_score`. A brand vetting 50
  creators gets it done in minutes instead of days.
